In [ ]:
pip install ultralytics opencv-python pyserial


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import cv2
from ultralytics import YOLO
import serial
import time

# === CONFIG ===
model_path = r"C:\Users\duduts\Documents\GitHub\dspCPE4A2526\Final Project\runs\detect\train7\weights\best.pt"
esp_port = "COM3"
baud_rate = 115200
relay_on_command = b"ON\n"
relay_off_command = b"OFF\n"

ON_TIME = 3     # seconds helmet required
OFF_TIME = 3    # seconds no helmet required
MAX_GAP = 0.5   # allowed detection drop (seconds)

# === Connect to ESP32 ===
try:
    ser = serial.Serial(esp_port, baud_rate, timeout=1)
    time.sleep(2)
    print(f"✅ Connected to ESP32 on {esp_port}")
except:
    print("❌ Serial connection failed")
    ser = None

# === Load YOLO model ===
model = YOLO(model_path)

# === Webcam ===
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    exit("❌ Webcam error")

relay_on = False
helmet_timer = 0
no_helmet_timer = 0
last_frame_time = time.time()
last_detect_time = None

while True:
    ret, frame = cap.read()
    if not ret:
        break

    now = time.time()
    dt = now - last_frame_time
    last_frame_time = now

    results = model(frame, conf=0.7)
    helmet_detected = False

    for cls in results[0].boxes.cls:
        if results[0].names[int(cls)].lower() == "helmet":
            helmet_detected = True
            break

    # ===== LOGIC =====
    if helmet_detected:
        helmet_timer += dt
        no_helmet_timer = 0
        last_detect_time = now
    else:
        if last_detect_time and (now - last_detect_time) < MAX_GAP:
            pass  # allow short flicker
        else:
            no_helmet_timer += dt
            helmet_timer = 0

    # ===== RELAY CONTROL =====
    if helmet_timer >= ON_TIME and not relay_on:
        if ser:
            ser.write(relay_on_command)
        relay_on = True
        print("✅ Relay ON (helmet stable)")

    if no_helmet_timer >= OFF_TIME and relay_on:
        if ser:
            ser.write(relay_off_command)
        relay_on = False
        print("❌ Relay OFF (no helmet)")

    # ===== DISPLAY =====
    annotated = results[0].plot()
    status = "ON" if relay_on else "OFF"

    cv2.putText(
        annotated,
        f"Relay: {status} | Helmet: {helmet_timer:.1f}s | No Helmet: {no_helmet_timer:.1f}s",
        (10, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0) if relay_on else (0, 0, 255),
        2,
    )

    cv2.imshow("Helmet Detection", annotated)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
if ser:
    ser.close()


✅ Connected to ESP32 on COM3

0: 480x640 (no detections), 31.7ms
Speed: 1.6ms preprocess, 31.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 28.3ms
Speed: 1.0ms preprocess, 28.3ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.1ms
Speed: 1.0ms preprocess, 29.1ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.6ms
Speed: 0.9ms preprocess, 29.6ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.3ms
Speed: 0.9ms preprocess, 29.3ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.7ms
Speed: 0.9ms preprocess, 29.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 29.0ms
Speed: 0.9ms preprocess, 29.0ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 28.6ms
Spee

KeyboardInterrupt: 